<a href="https://colab.research.google.com/github/AnjaArcus/Generador-de-c-digo-L-tex-para-presentaciones-beamer-/blob/main/Generador_de_beamer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install --upgrade PyMuPDF google-generativeai

# Generar codigo latex (.tex)

In [8]:
import os
import fitz  # PyMuPDF
import google.generativeai as genai

class BeamerGenerator:
    def __init__(self, api_key):
        # Configuración de la API de Gemini
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel('gemini-flash-lite-latest')

    # PASO 1: Extraer información de los PDFs
    def extract_text_from_pdfs(self, pdf_paths):
        full_text = ""
        for path in pdf_paths:
            try:
                with fitz.open(path) as doc:
                    for page in doc:
                        full_text += page.get_text() + "\n"
            except FileNotFoundError:
                print(f"Error: No se pudo encontrar el archivo {path}.")
            except Exception as e:
                print(f"Error al leer {path}: {e}")
        return full_text

    # PASO 2: Procesar con IA y crear estructura de Frames
    def generate_latex_content(self, raw_text):
        # Usamos doble llave {{ }} para que el f-string no confunda la sintaxis de LaTeX
        prompt = f"""
        Analiza el siguiente texto y genera una estructura de diapositivas para LaTeX Beamer.
        Usa solo el entorno \\begin{{frame}} ... \\end{{frame}}.
        Resume los puntos clave en itemize y usando    \\begin{{block}}{{Resumen Conceptual}} \\end{{exampleblock}} para conceptos o ideas importantes.
        Asegúrate de que el contenido sea académico y preciso. Para mejorar el resultado usar dos columnas en algunos frame.

        Texto a analizar:
        {raw_text[:8000]}
        """

        try:
            response = self.model.generate_content(prompt)
            # Limpiamos los posibles bloques de markdown (```latex ... ```) que a veces devuelve la API
            clean_text = response.text.replace("```latex\n", "").replace("```", "")
            return clean_text
        except Exception as e:
            print(f"Error al generar contenido con la API: {e}")
            return ""

    # PASO 3 & 4: Rellenar metadatos e insertar en la plantilla personalizada
    def finalize_latex(self, template_path, content_frames, metadata):
        try:
            with open(template_path, 'r', encoding='utf-8') as f:
                template = f.read()

            # Reemplazo de marcadores de posición (Placeholders)
            final_code = template.replace("{{LECCION}}", metadata.get('leccion', ''))
            final_code = final_code.replace("{{TITULO}}", metadata.get('titulo', ''))
            final_code = final_code.replace("{{CURSO}}", metadata.get('curso', ''))
            final_code = final_code.replace("{{AUTOR}}", metadata.get('autor', ''))

            # Insertar los frames en el cuerpo de la plantilla
            final_code = final_code.replace("{{CONTENIDO}}", content_frames)

            return final_code
        except FileNotFoundError:
            print(f"Error: No se encontró la plantilla {template_path}.")
            return ""


# --- FLUJO DE EJECUCIÓN ---
if __name__ == "__main__":
    # Configuración inicial (REEMPLAZA ESTO CON TU API KEY REAL)
    API_KEY = "REEMPLAZA ESTO CON TU API KEY REAL"
    generator = BeamerGenerator(API_KEY)

    # 1. Rutas de archivos (Asegúrate de que existan en la misma carpeta que este script)
    archivos_pdf = ["fotosintesis.pdf", "Guiadeestudio-fotosintesis.pdf"] # Puedes agregar más separados por comas
    plantilla_tex = "mi_plantilla.tex"

    # 2. Ejecución de pasos
    print("Extrayendo texto de los PDFs...")
    texto_extraido = generator.extract_text_from_pdfs(archivos_pdf)

    if texto_extraido:
        print("Generando estructura de Beamer con IA... (Esto puede tomar unos segundos)")
        frames_latex = generator.generate_latex_content(texto_extraido)

        if frames_latex:
            # 3. Datos específicos (Input del usuario)
            print("\n--- Ingrese los metadatos de la clase ---")
            datos = {
                'leccion': input("Número de lección: "),
                'titulo': input("Título: "),
                'curso': input("Nombre del curso: "),
                'autor': input("Nombre del autor: ")
            }

            # 4 & 5. Integración y Output
            print("\nEnsamblando el archivo LaTeX final...")
            codigo_final = generator.finalize_latex(plantilla_tex, frames_latex, datos)

            if codigo_final:
                output_filename = "presentacion_final.tex"  # Cambiar el nombre del archivo final
                with open(output_filename, "w", encoding='utf-8') as f:
                    f.write(codigo_final)
                print(f"\n¡Éxito! El código LaTeX completo se ha guardado en '{output_filename}'.")
            else:
                print("No se pudo generar el código final debido a un error con la plantilla.")
        else:
            print("No se generó contenido. Revisa tu conexión o tu API Key.")
    else:
        print("No se extrajo texto. Verifica las rutas de tus archivos PDF.")

Extrayendo texto de los PDFs...
Generando estructura de Beamer con IA... (Esto puede tomar unos segundos)

--- Ingrese los metadatos de la clase ---
Número de lección: 01
Título: Fotosintesis
Nombre del curso: Economía I
Nombre del autor: Anjaly Arcos Huaman

Ensamblando el archivo LaTeX final...

¡Éxito! El código LaTeX completo se ha guardado en 'presentacion_final.tex'.
